# FAHR-FaceAge 얼굴 나이 추정 (사진 폴더 → CSV)

얼굴 붓기 지표 프로젝트 **트랙 B**용 노트북입니다. 사진 폴더를 올리면 각 사진의 얼굴 나이를 추정해 `파일명, 추정 나이`를 CSV로 저장합니다.

**처리 순서** (FAHR-Face 저장소 README와 Hugging Face 모델 카드 기준)
1. RetinaFace(`retina-face==0.0.17`, 저장소 `other/face_crop_align.py`와 동일)로 얼굴 검출 → **얼굴이 정확히 1개인 사진만** 사용
2. `RetinaFace.extract_faces(align=True)`로 눈 위치 기준 회전 정렬 + 크롭
3. 112×112 RGB로 리사이즈, ImageNet 평균·표준편차로 정규화
4. `fahr_faceage.pth`(Hugging Face `fridolin22/FAHR-Face`)로 나이 회귀

**사용 방법**: 메뉴 `런타임 → 모두 실행`. GPU가 없어도 됩니다(사진 1장당 수 초).

**사진 준비 요령**
- JPG/PNG만 지원합니다(아이폰 HEIC는 JPG로 변환).
- 조건별로 **하위 폴더**를 만들어 zip으로 올리면 폴더 이름이 `condition` 열이 되어 조건별 평균·SD를 자동으로 계산합니다. 예: `standard/`, `casual_selfie/`, `morning/`, `evening/`, `lamp_warm/`
- 붓기 앱의 "표준 사진 저장"으로 받은 파일(`ff_...jpg`)은 파일명에서 시각·턱 높이·좌우·거리·안경 여부를 읽어 열로 추가합니다.

**주의**
- 모델 가중치 라이선스: **CC BY-NC-SA 4.0 (비상업·학술 연구용)**. 코드: Apache 2.0. 상용 앱에는 그대로 쓸 수 없습니다.
- 공개 벤치마크(APPA-REAL)에서 평균 절대오차 약 5.1세로 보고된 모델입니다. 여기서는 절대값보다 **같은 사람 반복 측정의 흔들림(SD)**을 보는 것이 목적입니다.
- Colab에 올린 사진은 Google 서버의 임시 런타임에 저장되며 세션이 끝나면 지워집니다. **IRB 승인 전에는 본인 사진만** 사용하세요.

## 1. 환경 준비

In [ ]:
# FAHR-Face 저장소(모델 클래스) + 전처리 패키지 설치
import os, sys, subprocess
if not os.path.exists("/content/FAHR-Face"):
    !git clone -q https://github.com/AIM-Harvard/FAHR-Face.git /content/FAHR-Face
# 저장소 전처리 환경과 같은 RetinaFace 버전. TensorFlow 2.16+에서는 tf-keras가 필요합니다.
!pip -q install "retina-face==0.0.17" tf-keras huggingface_hub
try:
    import tensorflow  # Colab에는 기본 설치되어 있음
except ImportError:
    !pip -q install tensorflow
sys.path.insert(0, "/content/FAHR-Face")

import torch, transformers, tensorflow as tf
print("torch", torch.__version__, "| transformers", transformers.__version__, "| tensorflow", tf.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 2. 모델 가중치 받기 (`fahr_faceage.pth`, 로그인 불필요)

In [ ]:
from huggingface_hub import hf_hub_download
CKPT = hf_hub_download("fridolin22/FAHR-Face", "fahr_faceage.pth")
print(CKPT, f"{os.path.getsize(CKPT)/1e6:.0f} MB")

## 3. 분석 함수

In [ ]:
import os, re, glob, zipfile
import numpy as np, pandas as pd, torch
from PIL import Image
from torchvision import transforms
from transformers import ViTConfig
from ViTForAgeEstimation import ViTForAgeRegression   # FAHR-Face 저장소의 원본 모델 클래스
from retinaface import RetinaFace

IMG_EXT = (".jpg", ".jpeg", ".png")
TRANSFORM = transforms.Compose([
    transforms.Resize((112, 112)),          # README/model card: 112x112 RGB
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
APP_RE = re.compile(r"^ff_(?P<ts>\d{8}T\d{6})_(?P<session>[a-z]+)_pitch(?P<pitch>-?[\d.]+)_yaw(?P<yaw>-?[\d.]+)_roll(?P<roll>-?[\d.]+)_dist(?P<dist>[\d.]+)_gl(?P<glasses>[01])")

# 체크포인트는 transformers 4.x의 ViT 층 이름으로 저장되어 있음. 최신 transformers(5.x)는 층 이름이 바뀌었으므로
# 현재 설치된 버전의 이름에 맞춰 변환한다(가중치 값은 그대로).
_OLD2NEW = [
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.attention\.query\.", r"vit.layers.\1.attention.q_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.attention\.key\.",   r"vit.layers.\1.attention.k_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.attention\.value\.", r"vit.layers.\1.attention.v_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.output\.dense\.",   r"vit.layers.\1.attention.o_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.intermediate\.dense\.",         r"vit.layers.\1.mlp.fc1."),
    (r"^vit\.encoder\.layer\.(\d+)\.output\.dense\.",               r"vit.layers.\1.mlp.fc2."),
    (r"^vit\.encoder\.layer\.(\d+)\.layernorm_",                      r"vit.layers.\1.layernorm_"),
]
def _convert_keys(sd, model_keys):
    if all(k in model_keys for k in sd):           # 이미 이름이 같음(구버전 transformers)
        return sd
    out = {}
    for k, v in sd.items():
        nk = k
        for pat, rep in _OLD2NEW:
            nk2 = re.sub(pat, rep, nk)
            if nk2 != nk: nk = nk2; break
        out[nk] = v
    return out

def load_age_model(ckpt_path, device):
    config = ViTConfig(image_size=112, num_hidden_layers=12, num_attention_heads=12)
    model = ViTForAgeRegression(config)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    sd = _convert_keys(sd, set(model.state_dict().keys()))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    bad_missing = [k for k in missing if not k.endswith("position_ids")]
    bad_unexp = [k for k in unexpected if not k.endswith("position_ids")]
    if bad_missing or bad_unexp:
        raise RuntimeError(f"체크포인트와 모델 구조가 맞지 않습니다. missing={bad_missing[:5]} unexpected={bad_unexp[:5]}")
    return model.to(device).eval()

def list_images(root):
    out = []
    for p in sorted(glob.glob(os.path.join(root, "**", "*"), recursive=True)):
        if p.lower().endswith(IMG_EXT) and "/__MACOSX/" not in p and not os.path.basename(p).startswith("._"):
            out.append(p)
    return out

def detect_crop(path, threshold=0.9):
    """README 방식: 얼굴이 정확히 1개일 때만 RetinaFace로 정렬(align=True) 후 크롭."""
    resp = RetinaFace.detect_faces(path, threshold=threshold)
    n = len(resp) if isinstance(resp, dict) else 0
    if n != 1:
        return None, n, None
    face = list(resp.values())[0]
    faces = RetinaFace.extract_faces(img_path=path, threshold=threshold, align=True)
    if not faces or faces[0].size == 0:
        return None, n, face
    return Image.fromarray(np.ascontiguousarray(faces[0])), n, face   # extract_faces returns RGB

@torch.no_grad()
def predict(model, pil_img, device):
    x = TRANSFORM(pil_img.convert("RGB")).unsqueeze(0).to(device)
    out = model(x)
    return float(out.age_output.reshape(-1)[0].item())

def parse_meta(path, root):
    name = os.path.basename(path)
    rel = os.path.relpath(os.path.dirname(path), root)
    meta = {"file": name, "condition": "" if rel == "." else rel.replace(os.sep, "/")}
    m = APP_RE.match(name)
    if m:
        d = m.groupdict()
        meta.update({"app_timestamp": d["ts"], "app_session": d["session"], "app_pitch": float(d["pitch"]),
                     "app_yaw": float(d["yaw"]), "app_roll": float(d["roll"]), "app_dist": float(d["dist"]),
                     "app_glasses": int(d["glasses"])})
    return meta

def run_folder(root, model, device, crop_dir=None, threshold=0.9, log=print):
    rows = []
    paths = list_images(root)
    for i, p in enumerate(paths, 1):
        meta = parse_meta(p, root)
        try:
            face, n, info = detect_crop(p, threshold)
            if face is None:
                meta.update({"status": "skipped", "reason": "얼굴 없음" if n == 0 else (f"얼굴 {n}개" if n > 1 else "크롭 실패"), "n_faces": n})
            else:
                meta.update({"status": "ok", "n_faces": 1, "predicted_age": round(predict(model, face, device), 2),
                             "det_score": round(float(info.get("score", np.nan)), 4),
                             "face_w_px": int(info["facial_area"][2] - info["facial_area"][0]),
                             "face_h_px": int(info["facial_area"][3] - info["facial_area"][1])})
                if crop_dir:
                    sub = os.path.join(crop_dir, meta["condition"]) if meta["condition"] else crop_dir
                    os.makedirs(sub, exist_ok=True)
                    face.save(os.path.join(sub, os.path.splitext(meta["file"])[0] + ".png"))
        except Exception as e:
            meta.update({"status": "error", "reason": str(e)[:200]})
        rows.append(meta)
        log(f"[{i}/{len(paths)}] {meta['file']}: {meta.get('predicted_age', meta.get('reason'))}")
    cols = ["file", "condition", "predicted_age", "status", "reason", "n_faces", "det_score", "face_w_px", "face_h_px",
            "app_timestamp", "app_session", "app_pitch", "app_yaw", "app_roll", "app_dist", "app_glasses"]
    df = pd.DataFrame(rows)
    return df[[c for c in cols if c in df.columns]]

def summarize(df):
    ok = df[df["status"] == "ok"]
    if ok.empty: return pd.DataFrame()
    key = "condition" if ok["condition"].astype(bool).any() else None
    g = ok.groupby(key) if key else ok.assign(condition="(전체)").groupby("condition")
    s = g["predicted_age"].agg(n="count", mean="mean", sd="std", min="min", max="max").round(2)
    s["range"] = (s["max"] - s["min"]).round(2)
    return s.reset_index()


# 사진이 너무 작으면(얼굴 폭 < 112px) 업스케일된 얼굴로 추정하게 되어 신뢰도가 떨어집니다.
MIN_FACE_PX = 112
MODEL = load_age_model(CKPT, DEVICE)
print("FAHR-FaceAge 로드 완료")

## 4. 사진 올리기

- `INPUT_MODE = "upload"`: 실행하면 파일 선택 창이 뜹니다. **zip 1개**(하위 폴더 가능) 또는 **사진 여러 장**을 고르세요.
- `INPUT_MODE = "drive"`: Google Drive 폴더 경로를 `DRIVE_FOLDER`에 적으세요. (예: `/content/drive/MyDrive/faceage_photos`)

In [ ]:
INPUT_MODE = "upload"   # @param ["upload", "drive"]
DRIVE_FOLDER = "/content/drive/MyDrive/faceage_photos"   # @param {type:"string"}

import shutil, zipfile
PHOTO_DIR = "/content/photos"
if INPUT_MODE == "upload":
    from google.colab import files
    shutil.rmtree(PHOTO_DIR, ignore_errors=True); os.makedirs(PHOTO_DIR)
    uploaded = files.upload()
    for name, data in uploaded.items():
        dst = os.path.join(PHOTO_DIR, name)
        with open(dst, "wb") as f: f.write(data)
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(dst) as z: z.extractall(PHOTO_DIR)
            os.remove(dst)
    # zip이 최상위 폴더 하나로 감싸져 있으면 그 안을 기준으로
    entries = [e for e in os.listdir(PHOTO_DIR) if e != "__MACOSX"]
    if len(entries) == 1 and os.path.isdir(os.path.join(PHOTO_DIR, entries[0])):
        sub = os.path.join(PHOTO_DIR, entries[0])
        if any(os.path.isdir(os.path.join(sub, e)) for e in os.listdir(sub)): PHOTO_DIR = sub
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PHOTO_DIR = DRIVE_FOLDER

imgs = list_images(PHOTO_DIR)
print(f"사진 {len(imgs)}장 발견:", PHOTO_DIR)
for p in imgs[:10]: print(" ", os.path.relpath(p, PHOTO_DIR))

## 5. 얼굴 나이 추정 실행

In [ ]:
CROP_DIR = "/content/aligned_faces"
shutil.rmtree(CROP_DIR, ignore_errors=True)
DETECTION_THRESHOLD = 0.9   # @param {type:"number"}

df = run_folder(PHOTO_DIR, MODEL, DEVICE, crop_dir=CROP_DIR, threshold=DETECTION_THRESHOLD)
if "face_w_px" in df.columns:
    df["small_face_warning"] = df["face_w_px"].lt(MIN_FACE_PX) & df["status"].eq("ok")

OUT_CSV = "/content/faceage_results.csv"
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("저장:", OUT_CSV)
print(df["status"].value_counts().to_string())
display(df)

## 6. 조건별 요약 (같은 사람 반복 촬영의 평균·SD)

In [ ]:
summary = summarize(df)
summary.to_csv("/content/faceage_summary.csv", index=False, encoding="utf-8-sig")
display(summary)
if df.get("small_face_warning") is not None and df["small_face_warning"].any():
    print("⚠ 얼굴 폭이 112px보다 작은 사진:", df.loc[df["small_face_warning"], "file"].tolist())

## 7. 정렬된 얼굴 확인 (전처리가 제대로 됐는지 눈으로 점검)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
ok = df[df["status"] == "ok"].head(24)
if len(ok):
    cols = 6; rows = (len(ok) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.4))
    for ax in axes.ravel(): ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), ok.iterrows()):
        sub = os.path.join(CROP_DIR, r["condition"]) if r["condition"] else CROP_DIR
        ax.imshow(Image.open(os.path.join(sub, os.path.splitext(r["file"])[0] + ".png")).resize((112, 112)))
        ax.set_title(f'{r["predicted_age"]:.1f}세\n{(r["condition"] or "")[:12]}', fontsize=8)
    plt.tight_layout(); plt.show()

## 8. 결과 내려받기

In [ ]:
from google.colab import files
files.download(OUT_CSV)
files.download("/content/faceage_summary.csv")
# 정렬된 얼굴 이미지도 받으려면 아래 두 줄의 주석을 푸세요 (얼굴 사진이므로 보관에 주의)
# shutil.make_archive("/content/aligned_faces", "zip", CROP_DIR)
# files.download("/content/aligned_faces.zip")

## 출력 열 설명

| 열 | 의미 |
|---|---|
| `file` | 파일명 |
| `condition` | 하위 폴더 이름(조건) |
| `predicted_age` | FAHR-FaceAge 추정 나이(세) |
| `status` / `reason` | `ok`, `skipped`(얼굴 0개 또는 2개 이상), `error` |
| `det_score`, `face_w_px`, `face_h_px` | RetinaFace 검출 점수, 원본에서의 얼굴 크기 |
| `small_face_warning` | 얼굴 폭 112px 미만 |
| `app_*` | 붓기 앱 표준 사진 파일명에서 읽은 촬영 시각·시점(am/pm/etc)·턱 높이·좌우·기울기·거리(얼굴 폭 %)·안경 |

**참고**: 저장소의 전처리 환경은 Python 3.8 + `retina-face 0.0.17`이고, 모델 카드의 추론 예시는 Python 3.11 + `torch 2.1.2` + `transformers 4.38.2`에서 시험되었습니다. 이 노트북은 Colab 기본 torch·transformers를 씁니다. 체크포인트는 transformers 4.x의 층 이름으로 저장되어 있어, 5.x에서는 3단계에서 층 이름을 자동 변환합니다(4.38.2로 만든 모델과 5.22.2에서 변환해 불러온 모델의 출력 차이 < 1e-6로 확인). 그래도 구조 오류가 나면 1단계 셀에 `!pip -q install "transformers==4.38.2"`를 추가하고 런타임을 다시 시작하세요.